# Coastal flood step 09: geographical mangrove attribution analysis

This notebook quantifies where mangrove-attributed avoided EAD is located, for minimum and maximum scenarios.

Outputs include:
- parish-level totals and area shares
- north/south and east/west summaries
- ranked positive and negative attribution hotspots
- scenario comparison tables

In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import FuncFormatter

jamaica_metric_grid_crs = "EPSG:3448"
mangrove_attribution_buffer_m = 5000

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

robyn_libraries_path = (base_path / "robyns_libraries").resolve()
if str(robyn_libraries_path) not in sys.path:
    sys.path.append(str(robyn_libraries_path))
import Robyn_paper_2_defs

scenario_paths = {
    "minimum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario/damage_estimates",
    "maximum": base_path / "dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario/damage_estimates",
}

parish_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jam_adm_shp/jam_admbnda_adm1.shp"
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"

out_dir = base_path / "dphil_paper_3/results_coastal_scenario_comparison/geographical_mangrove_attribution"
out_dir.mkdir(parents=True, exist_ok=True)

print("Output folder:", out_dir)
print("Mangrove attribution buffer (m):", mangrove_attribution_buffer_m)



def make_comma_tick_formatter(max_abs_value: float):
    if max_abs_value >= 100:
        decimals = 0
    elif max_abs_value >= 10:
        decimals = 1
    elif max_abs_value >= 1:
        decimals = 2
    else:
        decimals = 3

    tolerance = 0.5 * (10 ** (-decimals))

    def _format_tick(tick_value, _tick_position):
        tick_numeric = float(tick_value)
        if abs(tick_numeric) < tolerance:
            tick_numeric = 0.0
        return f"{tick_numeric:,.{decimals}f}"

    return FuncFormatter(_format_tick)



In [ ]:
parishes = gpd.read_file(parish_path).to_crs(jamaica_metric_grid_crs)
parishes = parishes[["NAME_1", "geometry"]].rename(columns={"NAME_1": "ParishName"})

jamaica_boundary = gpd.read_file(jamaica_boundary_path).to_crs(jamaica_metric_grid_crs)
jamaica_boundary_union = jamaica_boundary.geometry.union_all()
xmin, ymin, xmax, ymax = jamaica_boundary_union.bounds
xmid = (xmin + xmax) / 2.0
ymid = (ymin + ymax) / 2.0

print("Parishes loaded:", len(parishes))
print("Island midpoint (x, y):", round(xmid, 1), round(ymid, 1))

In [ ]:
def load_mangrove_attribution_gdf(damage_estimates_path: Path):
    gpkg = damage_estimates_path / f"mangrove_attribution/mangrove_attribution_total_{mangrove_attribution_buffer_m}m.gpkg"
    if not gpkg.exists():
        raise FileNotFoundError(f"Missing mangrove attribution file: {gpkg}")
    gdf = gpd.read_file(gpkg).to_crs(jamaica_metric_grid_crs)
    value_col = "Total_Avoided_EAD_USD_attributed"
    if value_col not in gdf.columns:
        raise KeyError(f"Missing column {value_col} in {gpkg}")
    gdf[value_col] = pd.to_numeric(gdf[value_col], errors="coerce").fillna(0.0)
    gdf["MangroveArea_ha"] = gdf.geometry.area / 10000.0
    c = gdf.geometry.centroid
    gdf["centroid_x"] = c.x
    gdf["centroid_y"] = c.y
    gdf["NS_Side"] = np.where(gdf["centroid_y"] >= ymid, "North", "South")
    gdf["EW_Side"] = np.where(gdf["centroid_x"] >= xmid, "East", "West")
    gdf["EffectSign"] = np.select(
        [gdf[value_col] > 0, gdf[value_col] < 0],
        ["ReducesDamage", "IncreasesDamage"],
        default="ZeroAttribution",
    )
    return gdf


scenario_gdfs = {
    scenario_name: load_mangrove_attribution_gdf(damage_estimates_path)
    for scenario_name, damage_estimates_path in scenario_paths.items()
}

for scenario_name, gdf in scenario_gdfs.items():
    print(scenario_name, "rows:", len(gdf), "area_ha:", round(gdf["MangroveArea_ha"].sum(), 3))



In [ ]:
def summarize_by_parish(scenario_name: str, gdf: gpd.GeoDataFrame):
    points = gpd.GeoDataFrame(
        gdf.drop(columns=["geometry"]).copy(),
        geometry=gpd.points_from_xy(gdf["centroid_x"], gdf["centroid_y"]),
        crs=jamaica_metric_grid_crs,
    )
    joined = gpd.sjoin(points, parishes, how="left", predicate="within")
    joined["ParishName"] = joined["ParishName"].fillna("Unassigned")

    grouped = (
        joined.groupby("ParishName", as_index=False)
        .agg(
            MangroveArea_ha=("MangroveArea_ha", "sum"),
            Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", "sum"),
            Positive_Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", lambda s: s[s > 0].sum()),
            Negative_Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", lambda s: s[s < 0].sum()),
            Area_ReducesDamage_ha=("MangroveArea_ha", lambda s: s[joined.loc[s.index, "EffectSign"] == "ReducesDamage"].sum()),
            Area_IncreasesDamage_ha=("MangroveArea_ha", lambda s: s[joined.loc[s.index, "EffectSign"] == "IncreasesDamage"].sum()),
            Area_ZeroAttribution_ha=("MangroveArea_ha", lambda s: s[joined.loc[s.index, "EffectSign"] == "ZeroAttribution"].sum()),
            MangroveCount=("Mangrove_ID", "nunique"),
        )
    )

    total_area = gdf["MangroveArea_ha"].sum()
    grouped["PctOfTotalMangroveArea"] = np.where(total_area > 0, 100.0 * grouped["MangroveArea_ha"] / total_area, np.nan)
    grouped["Scenario"] = scenario_name
    grouped = grouped.sort_values("Avoided_EAD_USD", ascending=False).reset_index(drop=True)
    return grouped


parish_summaries = {
    scenario_name: summarize_by_parish(scenario_name, gdf)
    for scenario_name, gdf in scenario_gdfs.items()
}

display(parish_summaries["minimum"].head(10))
display(parish_summaries["maximum"].head(10))

In [ ]:
def summarize_cardinal(gdf: gpd.GeoDataFrame, group_col: str, scenario_name: str):
    grouped = (
        gdf.groupby(group_col, as_index=False)
        .agg(
            MangroveArea_ha=("MangroveArea_ha", "sum"),
            Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", "sum"),
            Positive_Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", lambda s: s[s > 0].sum()),
            Negative_Avoided_EAD_USD=("Total_Avoided_EAD_USD_attributed", lambda s: s[s < 0].sum()),
            Area_ReducesDamage_ha=("MangroveArea_ha", lambda s: s[gdf.loc[s.index, "EffectSign"] == "ReducesDamage"].sum()),
            Area_IncreasesDamage_ha=("MangroveArea_ha", lambda s: s[gdf.loc[s.index, "EffectSign"] == "IncreasesDamage"].sum()),
            Area_ZeroAttribution_ha=("MangroveArea_ha", lambda s: s[gdf.loc[s.index, "EffectSign"] == "ZeroAttribution"].sum()),
            MangroveCount=("Mangrove_ID", "nunique"),
        )
    )
    total_area = gdf["MangroveArea_ha"].sum()
    grouped["PctOfTotalMangroveArea"] = np.where(total_area > 0, 100.0 * grouped["MangroveArea_ha"] / total_area, np.nan)
    grouped["Scenario"] = scenario_name
    return grouped.sort_values("Avoided_EAD_USD", ascending=False).reset_index(drop=True)


ns_summaries = {
    scenario_name: summarize_cardinal(gdf, "NS_Side", scenario_name)
    for scenario_name, gdf in scenario_gdfs.items()
}
ew_summaries = {
    scenario_name: summarize_cardinal(gdf, "EW_Side", scenario_name)
    for scenario_name, gdf in scenario_gdfs.items()
}

display(ns_summaries["minimum"])
display(ns_summaries["maximum"])
display(ew_summaries["minimum"])
display(ew_summaries["maximum"])

In [ ]:
def top_positive_negative(gdf: gpd.GeoDataFrame, scenario_name: str, n: int = 20):
    cols = ["Mangrove_ID", "Parish", "HECTARES", "MangroveArea_ha", "Total_Avoided_EAD_USD_attributed", "NS_Side", "EW_Side"]
    cols = [c for c in cols if c in gdf.columns]
    top_pos = gdf.loc[gdf["Total_Avoided_EAD_USD_attributed"] > 0, cols].sort_values(
        "Total_Avoided_EAD_USD_attributed", ascending=False
    ).head(n).copy()
    top_neg = gdf.loc[gdf["Total_Avoided_EAD_USD_attributed"] < 0, cols].sort_values(
        "Total_Avoided_EAD_USD_attributed", ascending=True
    ).head(n).copy()
    top_pos["Scenario"] = scenario_name
    top_neg["Scenario"] = scenario_name
    return top_pos, top_neg


top_pos = {}
top_neg = {}
for scenario_name, gdf in scenario_gdfs.items():
    top_pos[scenario_name], top_neg[scenario_name] = top_positive_negative(gdf, scenario_name, n=20)

display(top_pos["minimum"].head(10))
display(top_neg["minimum"].head(10))
display(top_pos["maximum"].head(10))
display(top_neg["maximum"].head(10))

In [ ]:
# Scenario comparison tables
parish_compare = (
    parish_summaries["minimum"][
        ["ParishName", "Avoided_EAD_USD", "Positive_Avoided_EAD_USD", "Negative_Avoided_EAD_USD", "MangroveArea_ha", "Area_ReducesDamage_ha", "Area_IncreasesDamage_ha"]
    ]
    .rename(columns=lambda c: c if c == "ParishName" else f"Min_{c}")
    .merge(
        parish_summaries["maximum"][
            ["ParishName", "Avoided_EAD_USD", "Positive_Avoided_EAD_USD", "Negative_Avoided_EAD_USD", "MangroveArea_ha", "Area_ReducesDamage_ha", "Area_IncreasesDamage_ha"]
        ].rename(columns=lambda c: c if c == "ParishName" else f"Max_{c}"),
        on="ParishName",
        how="outer",
    )
)
parish_compare["Delta_Avoided_EAD_USD_MaxMinusMin"] = parish_compare["Max_Avoided_EAD_USD"] - parish_compare["Min_Avoided_EAD_USD"]
parish_compare = parish_compare.sort_values("Delta_Avoided_EAD_USD_MaxMinusMin", ascending=False).reset_index(drop=True)

display(parish_compare.head(14))

In [ ]:
# Quick parish map for each scenario (green positive, red negative)
def make_parish_map(parish_summary_df: pd.DataFrame, scenario_name: str):
    map_df = parishes.merge(parish_summary_df[["ParishName", "Avoided_EAD_USD"]], on="ParishName", how="left")
    map_df["Avoided_EAD_USD"] = map_df["Avoided_EAD_USD"].fillna(0.0)
    vmax = float(np.nanmax(np.abs(map_df["Avoided_EAD_USD"])))
    if vmax == 0:
        vmax = 1.0

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6)
    map_df.plot(
        ax=ax,
        column="Avoided_EAD_USD",
        cmap="RdYlGn",
        norm=norm,
        linewidth=0.2,
        edgecolor="gray",
        legend=False,
    )

    scalar_mappable = ScalarMappable(norm=norm, cmap="RdYlGn")
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(
        scalar_mappable,
        ax=ax,
        orientation="horizontal",
        fraction=0.036,
        pad=0.008,
    )
    colorbar.set_label("Mangrove-attributed avoided EAD (USD) by parish")
    colorbar.ax.xaxis.set_major_formatter(
        make_comma_tick_formatter(vmax)
    )

    Robyn_paper_2_defs.draw_scale_bar(
        ax,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(
        ax,
        location=(0.88, 0.86),
        size=0.05,
        fontsize=8,
        label_offset=0.02,
    )

    ax.set_title(f"Parish attribution totals - {scenario_name}")
    ax.set_axis_off()

    fig.subplots_adjust(top=0.94, bottom=0.09)
    out_png = out_dir / f"parish_mangrove_attribution_{scenario_name}.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)


make_parish_map(parish_summaries["minimum"], "minimum")
make_parish_map(parish_summaries["maximum"], "maximum")




In [ ]:
# Save outputs
parish_summaries["minimum"].to_csv(out_dir / "parish_summary_minimum.csv", index=False)
parish_summaries["maximum"].to_csv(out_dir / "parish_summary_maximum.csv", index=False)
ns_summaries["minimum"].to_csv(out_dir / "north_south_summary_minimum.csv", index=False)
ns_summaries["maximum"].to_csv(out_dir / "north_south_summary_maximum.csv", index=False)
ew_summaries["minimum"].to_csv(out_dir / "east_west_summary_minimum.csv", index=False)
ew_summaries["maximum"].to_csv(out_dir / "east_west_summary_maximum.csv", index=False)
top_pos["minimum"].to_csv(out_dir / "top20_positive_mangrove_attribution_minimum.csv", index=False)
top_neg["minimum"].to_csv(out_dir / "top20_negative_mangrove_attribution_minimum.csv", index=False)
top_pos["maximum"].to_csv(out_dir / "top20_positive_mangrove_attribution_maximum.csv", index=False)
top_neg["maximum"].to_csv(out_dir / "top20_negative_mangrove_attribution_maximum.csv", index=False)
parish_compare.to_csv(out_dir / "parish_summary_minimum_vs_maximum.csv", index=False)

print("Saved geographical mangrove attribution outputs to:", out_dir)

## Catchment analysis using river major basins

Uses the same major catchments as river flood analysis, without modifying that input file.
Mangrove-attributed EAD is apportioned to catchments by intersection area fraction.


In [ ]:
catchments_path = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
if not catchments_path.exists():
    raise FileNotFoundError(f"Missing catchments file: {catchments_path}")

catchments = gpd.read_file(catchments_path).to_crs(jamaica_metric_grid_crs)
catchments = catchments[["catchment_uid", "area_m2", "area_km2", "area_ha", "geometry"]].copy()
catchments["catchment_uid"] = pd.to_numeric(catchments["catchment_uid"], errors="coerce")
catchments = catchments[catchments["catchment_uid"].notna()].copy()
catchments["catchment_uid"] = catchments["catchment_uid"].astype(int)

print("Catchments loaded:", len(catchments))
display(catchments[["catchment_uid", "area_km2", "area_ha"]].head(10))


In [ ]:
def summarize_by_catchment_area_weighted(scenario_name: str, mangrove_gdf: gpd.GeoDataFrame, catchments_gdf: gpd.GeoDataFrame):
    value_col = "Total_Avoided_EAD_USD_attributed"
    src = mangrove_gdf[["Mangrove_ID", value_col, "geometry"]].copy()
    src["source_area_ha"] = src.geometry.area / 10000.0

    inter = gpd.overlay(
        src,
        catchments_gdf[["catchment_uid", "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )
    inter["inter_area_ha"] = inter.geometry.area / 10000.0
    inter = inter[inter["inter_area_ha"] > 0].copy()
    inter["area_fraction"] = np.where(
        inter["source_area_ha"] > 0,
        inter["inter_area_ha"] / inter["source_area_ha"],
        0.0,
    )
    inter["Avoided_EAD_USD_weighted"] = inter[value_col] * inter["area_fraction"]
    inter["Positive_Avoided_EAD_USD_weighted"] = np.where(inter["Avoided_EAD_USD_weighted"] > 0, inter["Avoided_EAD_USD_weighted"], 0.0)
    inter["Negative_Avoided_EAD_USD_weighted"] = np.where(inter["Avoided_EAD_USD_weighted"] < 0, inter["Avoided_EAD_USD_weighted"], 0.0)
    inter["Area_ReducesDamage_ha"] = np.where(inter["Avoided_EAD_USD_weighted"] > 0, inter["inter_area_ha"], 0.0)
    inter["Area_IncreasesDamage_ha"] = np.where(inter["Avoided_EAD_USD_weighted"] < 0, inter["inter_area_ha"], 0.0)
    inter["Area_ZeroAttribution_ha"] = np.where(inter["Avoided_EAD_USD_weighted"] == 0, inter["inter_area_ha"], 0.0)

    summary = (
        inter.groupby("catchment_uid", as_index=False)
        .agg(
            MangroveArea_ha=("inter_area_ha", "sum"),
            Avoided_EAD_USD=("Avoided_EAD_USD_weighted", "sum"),
            Positive_Avoided_EAD_USD=("Positive_Avoided_EAD_USD_weighted", "sum"),
            Negative_Avoided_EAD_USD=("Negative_Avoided_EAD_USD_weighted", "sum"),
            Area_ReducesDamage_ha=("Area_ReducesDamage_ha", "sum"),
            Area_IncreasesDamage_ha=("Area_IncreasesDamage_ha", "sum"),
            Area_ZeroAttribution_ha=("Area_ZeroAttribution_ha", "sum"),
            MangroveCount_Intersecting=("Mangrove_ID", "nunique"),
        )
    )

    summary = catchments_gdf[["catchment_uid", "area_km2", "area_ha"]].merge(summary, on="catchment_uid", how="left")
    fill_cols = [
        "MangroveArea_ha", "Avoided_EAD_USD", "Positive_Avoided_EAD_USD", "Negative_Avoided_EAD_USD",
        "Area_ReducesDamage_ha", "Area_IncreasesDamage_ha", "Area_ZeroAttribution_ha", "MangroveCount_Intersecting"
    ]
    for col in fill_cols:
        summary[col] = summary[col].fillna(0.0)

    total_mangrove_area = float(mangrove_gdf["MangroveArea_ha"].sum())
    summary["PctOfScenarioTotalMangroveArea"] = np.where(
        total_mangrove_area > 0,
        100.0 * summary["MangroveArea_ha"] / total_mangrove_area,
        np.nan,
    )
    summary["Scenario"] = scenario_name
    summary = summary.sort_values("Avoided_EAD_USD", ascending=False).reset_index(drop=True)
    return summary


catchment_summaries = {
    scenario_name: summarize_by_catchment_area_weighted(scenario_name, gdf, catchments)
    for scenario_name, gdf in scenario_gdfs.items()
}

display(catchment_summaries["minimum"].head(15))
display(catchment_summaries["maximum"].head(15))


In [ ]:
# Catchment comparison between scenarios
catchment_compare = (
    catchment_summaries["minimum"][
        ["catchment_uid", "Avoided_EAD_USD", "Positive_Avoided_EAD_USD", "Negative_Avoided_EAD_USD", "MangroveArea_ha", "Area_ReducesDamage_ha", "Area_IncreasesDamage_ha"]
    ]
    .rename(columns=lambda c: c if c == "catchment_uid" else f"Min_{c}")
    .merge(
        catchment_summaries["maximum"][
            ["catchment_uid", "Avoided_EAD_USD", "Positive_Avoided_EAD_USD", "Negative_Avoided_EAD_USD", "MangroveArea_ha", "Area_ReducesDamage_ha", "Area_IncreasesDamage_ha"]
        ].rename(columns=lambda c: c if c == "catchment_uid" else f"Max_{c}"),
        on="catchment_uid",
        how="outer",
    )
)
catchment_compare["Delta_Avoided_EAD_USD_MaxMinusMin"] = catchment_compare["Max_Avoided_EAD_USD"] - catchment_compare["Min_Avoided_EAD_USD"]
catchment_compare = catchment_compare.sort_values("Delta_Avoided_EAD_USD_MaxMinusMin", ascending=False).reset_index(drop=True)

display(catchment_compare.head(20))


In [ ]:
# Top catchments with strongest reductions and strongest increases
top_catchment_rows = []
for scenario_name, df in catchment_summaries.items():
    top_reduce = df.sort_values("Avoided_EAD_USD", ascending=False).head(10).copy()
    top_increase = df.sort_values("Avoided_EAD_USD", ascending=True).head(10).copy()
    top_reduce["RankType"] = "TopReduction"
    top_increase["RankType"] = "TopIncrease"
    top_reduce["Scenario"] = scenario_name
    top_increase["Scenario"] = scenario_name
    top_catchment_rows.extend([top_reduce, top_increase])

top_catchments = pd.concat(top_catchment_rows, ignore_index=True)
display(top_catchments.head(25))


In [ ]:
# Quick catchment maps for each scenario (green positive, red negative)
def make_catchment_map(catchment_summary_df: pd.DataFrame, scenario_name: str):
    map_df = catchments.merge(catchment_summary_df[["catchment_uid", "Avoided_EAD_USD"]], on="catchment_uid", how="left")
    map_df["Avoided_EAD_USD"] = map_df["Avoided_EAD_USD"].fillna(0.0)
    vmax = float(np.nanmax(np.abs(map_df["Avoided_EAD_USD"])))
    if vmax == 0:
        vmax = 1.0

    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, ax = plt.subplots(1, 1, figsize=(9, 8))
    jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.5)
    map_df.plot(
        ax=ax,
        column="Avoided_EAD_USD",
        cmap="RdYlGn",
        norm=norm,
        linewidth=0.2,
        edgecolor="gray",
        legend=False,
    )

    scalar_mappable = ScalarMappable(norm=norm, cmap="RdYlGn")
    scalar_mappable.set_array([])
    colorbar = fig.colorbar(
        scalar_mappable,
        ax=ax,
        orientation="horizontal",
        fraction=0.036,
        pad=0.008,
    )
    colorbar.set_label("Mangrove-attributed avoided EAD (USD) by catchment")
    colorbar.ax.xaxis.set_major_formatter(
        make_comma_tick_formatter(vmax)
    )

    Robyn_paper_2_defs.draw_scale_bar(
        ax,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(
        ax,
        location=(0.88, 0.86),
        size=0.05,
        fontsize=8,
        label_offset=0.02,
    )

    ax.set_title(f"Catchment attribution totals - {scenario_name}")
    ax.set_axis_off()

    fig.subplots_adjust(top=0.94, bottom=0.09)
    out_png = out_dir / f"catchment_mangrove_attribution_{scenario_name}.png"
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_png)


make_catchment_map(catchment_summaries["minimum"], "minimum")
make_catchment_map(catchment_summaries["maximum"], "maximum")




## Catchment min/max panel (shared scale, compact layout)

Single figure with minimum and maximum catchment attribution totals side-by-side,
with one shared horizontal colorbar underneath and automatically scaled units.


In [ ]:
def choose_unit_scale(max_abs_usd: float):
    if max_abs_usd >= 1_000:
        return 1_000.0, 'USD thousands'
    return 1.0, 'USD'


def draw_outlier_inset_table(ax, outlier_df: pd.DataFrame, threshold_scaled: float, unit_label: str, y_anchor: float = 0.02):
    outlier_df = outlier_df.copy()
    if outlier_df.empty:
        ax.text(
            0.985, y_anchor,
            f'Outliers > {threshold_scaled:g}\nNone',
            transform=ax.transAxes,
            ha='right', va='bottom',
            fontsize=6.8,
            color='#2f2f2f',
            bbox={'facecolor': 'white', 'edgecolor': '#808080', 'alpha': 0.88, 'pad': 0.22},
            zorder=7,
        )
        return

    outlier_values = outlier_df[['catchment_uid', 'Avoided_EAD']].copy()
    outlier_values = outlier_values.sort_values('Avoided_EAD', key=lambda values: values.abs(), ascending=False)

    cell_text = [
        [str(int(row['catchment_uid'])), f"{float(row['Avoided_EAD']):,.1f}"]
        for _, row in outlier_values.iterrows()
    ]
    n_rows = len(cell_text)
    table_height = min(0.26, 0.06 + 0.06 * n_rows)

    table = ax.table(
        cellText=cell_text,
        colLabels=['uid', 'value'],
        cellLoc='center',
        colLoc='center',
        bbox=[0.74, y_anchor, 0.24, table_height],
        zorder=7,
    )
    table.auto_set_font_size(False)
    table.set_fontsize(6.3)

    for (row_index, column_index), table_cell in table.get_celld().items():
        if row_index == 0:
            table_cell.set_text_props(weight='bold')
        table_cell.set_facecolor((1.0, 1.0, 1.0, 0.88))
        table_cell.set_edgecolor('#7a7a7a')
        table_cell.set_linewidth(0.4)

    ax.text(
        0.74, y_anchor + table_height + 0.008,
        f'|value| > {threshold_scaled:g} ({unit_label})',
        transform=ax.transAxes,
        ha='left', va='bottom',
        fontsize=6.6,
        color='#2f2f2f',
        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.8, 'pad': 0.15},
        zorder=7,
    )


min_map_df = catchments.merge(
    catchment_summaries['minimum'][['catchment_uid', 'Avoided_EAD_USD']],
    on='catchment_uid',
    how='left',
)
max_map_df = catchments.merge(
    catchment_summaries['maximum'][['catchment_uid', 'Avoided_EAD_USD']],
    on='catchment_uid',
    how='left',
)

for map_df in [min_map_df, max_map_df]:
    map_df['Avoided_EAD_USD'] = pd.to_numeric(map_df['Avoided_EAD_USD'], errors='coerce').fillna(0.0)

true_shared_abs_max_usd = float(max(
    min_map_df['Avoided_EAD_USD'].abs().max(),
    max_map_df['Avoided_EAD_USD'].abs().max(),
))
if true_shared_abs_max_usd <= 0:
    true_shared_abs_max_usd = 1.0

unit_scale, unit_label = choose_unit_scale(true_shared_abs_max_usd)

for map_df in [min_map_df, max_map_df]:
    map_df['_plot_val'] = map_df['Avoided_EAD_USD'] / unit_scale

# Robust shared cap to improve contrast in the bulk of catchments.
display_quantile = 0.98
all_abs_scaled = pd.concat([
    min_map_df['_plot_val'].abs(),
    max_map_df['_plot_val'].abs(),
], ignore_index=True)
shared_abs_display = float(all_abs_scaled.quantile(display_quantile))
if shared_abs_display <= 0:
    shared_abs_display = true_shared_abs_max_usd / unit_scale

# Outlier threshold in displayed units (e.g., 200 for USD thousands).
outlier_threshold_scaled = 200.0

# Use white at zero for clearer near-zero interpretation.
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#b2182b', '#f7f7f7', '#1a9850'],
    N=256,
)
norm = TwoSlopeNorm(vmin=-shared_abs_display, vcenter=0.0, vmax=shared_abs_display)

fig, axes = plt.subplots(2, 1, figsize=(9.5, 9.3), sharex=True, sharey=True)

for axis, map_df, scenario_label, table_y_anchor in [
    (axes[0], min_map_df, 'minimum', 0.020),
    (axes[1], max_map_df, 'maximum', -0.025),
]:
    map_df.plot(
        ax=axis,
        column='_plot_val',
        cmap=cmap,
        norm=norm,
        linewidth=0.2,
        edgecolor='gray',
        legend=False,
        zorder=2,
    )
    jamaica_boundary.boundary.plot(ax=axis, color='black', linewidth=0.45, zorder=3)

    representative_points = map_df.geometry.representative_point()
    for (_, row), point in zip(map_df.iterrows(), representative_points):
        axis.text(
            point.x,
            point.y,
            str(int(row['catchment_uid'])),
            fontsize=5.5,
            color='#4A4A4A',
            ha='center',
            va='center',
            zorder=4,
            alpha=0.95,
        )

    outliers = map_df[map_df['_plot_val'].abs() > outlier_threshold_scaled].copy()
    if not outliers.empty:
        outliers.boundary.plot(ax=axis, color='#3f007d', linewidth=1.55, zorder=5)
        outlier_points = outliers.geometry.representative_point()
        for (_, row), point in zip(outliers.iterrows(), outlier_points):
            axis.text(
                point.x,
                point.y,
                str(int(row['catchment_uid'])),
                fontsize=8,
                color='#3f007d',
                fontweight='bold',
                ha='center',
                va='center',
                bbox={'facecolor': 'white', 'edgecolor': '#3f007d', 'alpha': 0.85, 'pad': 0.22},
                zorder=6,
            )

    outlier_table_df = outliers[['catchment_uid', '_plot_val']].copy().rename(columns={'_plot_val': 'Avoided_EAD'})
    draw_outlier_inset_table(axis, outlier_table_df, outlier_threshold_scaled, unit_label, y_anchor=table_y_anchor)

    Robyn_paper_2_defs.draw_scale_bar(
        axis,
        location=(0.88, 0.78),
        length_km=20,
        linewidth=0.6,
        label_offset=0.02,
        km_offset=0.01,
    )
    Robyn_paper_2_defs.draw_north_arrow(
        axis,
        location=(0.88, 0.86),
        size=0.05,
        fontsize=8,
        label_offset=0.02,
    )

    axis.set_title(f'Catchment attribution totals - {scenario_label}', fontsize=11.8, pad=3)
    axis.set_axis_off()

pad_x = (xmax - xmin) * 0.018
pad_y = (ymax - ymin) * 0.008
for axis in axes:
    axis.set_xlim(xmin - pad_x, xmax + pad_x)
    axis.set_ylim(ymin - pad_y, ymax + pad_y)

cax = fig.add_axes([0.17, 0.075, 0.66, 0.022])
scalar_mappable = ScalarMappable(norm=norm, cmap=cmap)
scalar_mappable.set_array([])
colorbar = fig.colorbar(scalar_mappable, cax=cax, orientation='horizontal')
colorbar.set_label(f'Mangrove-attributed avoided EAD by catchment ({unit_label})')
colorbar.ax.xaxis.set_major_formatter(
    make_comma_tick_formatter(shared_abs_display)
)

fig.subplots_adjust(left=0.03, right=0.97, top=0.968, bottom=0.11, hspace=0.03)

out_png = out_dir / 'catchment_mangrove_attribution_minimum_maximum_panel_outliers.png'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
plt.show()

min_outliers_csv = out_dir / 'catchment_outliers_minimum_above_threshold.csv'
max_outliers_csv = out_dir / 'catchment_outliers_maximum_above_threshold.csv'

(
    min_map_df.loc[
        min_map_df['_plot_val'].abs() > outlier_threshold_scaled,
        ['catchment_uid', 'Avoided_EAD_USD', '_plot_val'],
    ]
    .rename(columns={'_plot_val': f'Avoided_EAD_{unit_label.replace(" ", "_")}'} )
    .sort_values('Avoided_EAD_USD', key=lambda values: values.abs(), ascending=False)
    .to_csv(min_outliers_csv, index=False)
)
(
    max_map_df.loc[
        max_map_df['_plot_val'].abs() > outlier_threshold_scaled,
        ['catchment_uid', 'Avoided_EAD_USD', '_plot_val'],
    ]
    .rename(columns={'_plot_val': f'Avoided_EAD_{unit_label.replace(" ", "_")}'} )
    .sort_values('Avoided_EAD_USD', key=lambda values: values.abs(), ascending=False)
    .to_csv(max_outliers_csv, index=False)
)

print('Unit scale:', unit_label)
print('Display cap:', f"±{shared_abs_display:,.2f} {unit_label} (q={display_quantile:.2f})")
print('Outlier threshold:', f"|value| > {outlier_threshold_scaled:g} {unit_label}")
print('Saved:', out_png)
print('Saved:', min_outliers_csv)
print('Saved:', max_outliers_csv)






In [ ]:
# Save catchment outputs
catchment_summaries["minimum"].to_csv(out_dir / "catchment_summary_minimum.csv", index=False)
catchment_summaries["maximum"].to_csv(out_dir / "catchment_summary_maximum.csv", index=False)
catchment_compare.to_csv(out_dir / "catchment_summary_minimum_vs_maximum.csv", index=False)
top_catchments.to_csv(out_dir / "catchment_top_reduction_increase_minimum_maximum.csv", index=False)

print("Saved catchment outputs to:", out_dir)
print("- catchment_summary_minimum.csv")
print("- catchment_summary_maximum.csv")
print("- catchment_summary_minimum_vs_maximum.csv")
print("- catchment_top_reduction_increase_minimum_maximum.csv")
